# Hutch benchmark data-quality audit

This notebook checks the repeated control-plane captures used in `docs/realignment/performance.md`. It verifies raw sample counts, finite values, machine/runtime comparability, the common harness digest, and the revision provenance before calculating medians and nearest-rank p90s.

The baseline is the source tree from Git archive `b3e4b15`; the identical benchmark harness was copied into that archive because the older revision did not retain raw samples. Its missing Git metadata and dirty flag are expected. The current capture is clean commit `bf09d97`. These local mock measurements do not cover model inference, providers, browser startup, or queueing.

In [ ]:
import json
import math
import statistics
from pathlib import Path

ROOT = Path.cwd()
BASELINE_PATH = ROOT / 'docs/realignment/artifacts/performance/baseline-b3e4b15.json'
CURRENT_PATH = ROOT / 'docs/realignment/artifacts/performance/current-bf09d97.json'
baseline = json.loads(BASELINE_PATH.read_text())
current = json.loads(CURRENT_PATH.read_text())
captures = {'baseline': baseline, 'current': current}
print('Loaded:', BASELINE_PATH, CURRENT_PATH)

In [ ]:
# Validate capture provenance and the retained raw samples.
assert baseline['meta']['source_revision'] == 'b3e4b15'
assert current['meta']['source_revision'] == 'bf09d97'
assert current['meta']['git_head'].startswith('bf09d97')
assert current['meta']['working_tree_dirty'] is False
assert baseline['meta']['git_head'] == 'unknown'  # Git archive has no .git directory
assert baseline['meta']['working_tree_dirty'] is True  # same-hash harness was injected into archive
for key in ('cpu', 'cores', 'python', 'platform', 'benchmark_harness_sha256', 'sample_schema', 'measured_on'):
    assert baseline['meta'][key] == current['meta'][key], (key, baseline['meta'][key], current['meta'][key])
assert baseline['meta']['benchmark_harness_sha256'] == '66dab3a98f9fbd7619977274442f287e82131a155bd678618775c58efd7babd6'
expected_counts = {'cli_version_ms': 11, 'cli_status_ms': 11, 'events_per_sec': 5, 'step_overhead_ms': 12,
                   'idle_rss_daemon_mb': 3, 'idle_cpu_daemon_s_per_hour': 3,
                   'idle_rss_serve_mb': 3, 'idle_cpu_serve_s_per_hour': 3}
for label, capture in captures.items():
    assert set(capture['samples']) == set(expected_counts)
    assert set(capture['distributions']) == set(expected_counts)
    for metric, expected_n in expected_counts.items():
        raw = capture['samples'][metric]
        dist = capture['distributions'][metric]
        assert len(raw) == expected_n == dist['n'], (label, metric, len(raw), dist['n'])
        assert all(math.isfinite(float(x)) for x in raw), (label, metric, raw)
        assert math.isclose(statistics.median(raw), dist['median'], rel_tol=1e-12, abs_tol=1e-12)
print('PASS: matching host, Python, harness digest, revisions, and all expected raw sample counts')

In [ ]:
def nearest_rank(values, q=0.90):
    ordered = sorted(float(v) for v in values)
    return ordered[math.ceil(q * len(ordered)) - 1]

def stats(capture, metric):
    values = capture['samples'][metric]
    return statistics.median(values), nearest_rank(values), len(values)

labels = {
    'cli_version_ms': 'CLI version startup (ms)',
    'cli_status_ms': 'CLI status (ms)',
    'events_per_sec': 'Event throughput (events/s)',
    'step_overhead_ms': 'Mock step overhead (ms)',
    'idle_rss_daemon_mb': 'Idle daemon RSS (MB)',
    'idle_rss_serve_mb': 'Idle serve RSS (MB)',
}
rows = []
for metric, label in labels.items():
    b50, b90, bn = stats(baseline, metric)
    c50, c90, cn = stats(current, metric)
    delta_pct = (c50 / b50 - 1) * 100
    rows.append((label, b50, b90, bn, c50, c90, cn, delta_pct))
for label, b50, b90, bn, c50, c90, cn, delta in rows:
    print(f'{label:28} baseline {b50:10.3f}/{b90:10.3f} n={bn:<2} current {c50:10.3f}/{c90:10.3f} n={cn:<2} p50 change {delta:+.1f}%')

## Interpretation limits

- Nearest-rank p90 is the maximum when `n` is 3 or 5, and remains a coarse tail indicator at `n=11`; none of these values is an SLA.
- Idle CPU uses an 8-second observation. One 10 ms clock tick corresponds to about 4.5 CPU-seconds/hour, so adjacent values cannot establish a meaningful CPU regression.
- The existing aggregate baseline files have no raw repetitions. They are historical context only; this comparison instead reran the same harness against the archived baseline source.
- Results describe one Linux host and local mock/control-plane paths. No live provider, inference, browser, or physical-device workload was measured.